# Cache port (`hobo:cache`) vs durable `kv` — ADR-0118

A cache is **ephemeral, TTL-first, evictable, and never a source of truth**; a durable `kv` is **authoritative**. Below, a cache-aside read stays correct across TTL expiry, eviction, and a full flush — because it always falls back to the authoritative `kv`.

In [1]:
import { cacheAside, MISS, SimCache, SimDurableKv } from "./src/cache-contract.ts";

let now = 0;
const clock = () => now;
const kv = new SimDurableKv<string>();
const cache = new SimCache<string>(clock);

kv.set("user:1", "Ada");
console.log("kv durable read:", kv.get("user:1"));

const first = cacheAside(cache, "user:1", () => kv.get("user:1"), 10);
console.log(`read#1 hit=${first.hit} value=${first.value}`);
const second = cacheAside(cache, "user:1", () => kv.get("user:1"), 10);
console.log(`read#2 hit=${second.hit} value=${second.value}`);

now += 11;
const afterTtl = cacheAside(cache, "user:1", () => kv.get("user:1"), 10);
console.log(`read#3 (post-TTL) hit=${afterTtl.hit} value=${afterTtl.value}`);

cache.evictAll();
const afterEvict = cacheAside(cache, "user:1", () => kv.get("user:1"), 10);
console.log(`read#4 (post-evict) hit=${afterEvict.hit} value=${afterEvict.value}`);

cache.evictAll();
console.log(`post-flush: cache=${cache.get("user:1") === MISS ? "MISS" : "?"} kv=${kv.get("user:1")}`);
console.log("cold key:", cache.get("absent") === MISS ? "MISS" : "ERR");
console.log("CONTRACT HELD: cache is ephemeral/TTL/evictable; kv is durable/authoritative");

kv durable read: Ada
read#1 hit=false value=Ada
read#2 hit=true value=Ada
read#3 (post-TTL) hit=false value=Ada
read#4 (post-evict) hit=false value=Ada
post-flush: cache=MISS kv=Ada
cold key: MISS
CONTRACT HELD: cache is ephemeral/TTL/evictable; kv is durable/authoritative
